In [0]:
# COMMAND ----------

import json
import time
import datetime as dt
from typing import Optional, Dict, Any

import requests

# COMMAND ----------

# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "fred_raw"

BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

ALL_SERIES: Dict[str, str] = {
    # Daily market signals
    "DCOILWTICO":           "Crude Oil Prices: WTI (Daily)",
    "DGS10":                "10-Year Treasury Constant Maturity Rate (Daily)",
    "FEDFUNDS":             "Effective Federal Funds Rate (Daily)",
    "T5YIE":                "5-Year Breakeven Inflation Rate (Daily)",
    "DEXUSEU":              "US/Euro FX Rate (Daily)",
    "DEXCAUS":              "Canada/US FX Rate (Daily)",
    # Weekly labor
    "ICSA":                 "Initial Claims (Weekly)",
    "IC4WSA":               "4-Week Moving Avg Initial Claims (Weekly)",
    # Monthly economic
    "MRTSSM443USS":         "Retail Sales: Electronics & Appliance Stores",
    "UMCSENT":              "Consumer Sentiment",
    "HOUST":                "Housing Starts",
    "DSPIC96":              "Real Disposable Personal Income",
    "WPU1171":              "PPI: Electronic Computer Manufacturing",
    "PCU33443344":          "PPI: Computers & Peripheral Equipment",
    "FRGSHPUSM649NCIS":     "Cass Freight Index: Shipments",
}

BACKFILL_START = "2010-01-01"
OVERLAP_DAYS = 3

# COMMAND ----------

# ---------------- Mode ----------------
dbutils.widgets.dropdown("MODE", "incremental", ["incremental", "backfill"])
MODE = dbutils.widgets.get("MODE").strip().lower()
print(f"MODE: {MODE}")

# Secret
SECRET_SCOPE = "fred-api-scope"
SECRET_KEY   = "fredkey"
FRED_API_KEY = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY)

# COMMAND ----------

# ---------------- Helpers ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def get_json_with_retry(
    url: str, params: Dict[str, Any],
    headers: Optional[Dict[str, str]] = None,
    retries: int = 4, timeout: int = 30
) -> Dict[str, Any]:
    backoff = 1.0
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt == retries:
                break
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"Request failed after {retries} attempts: {last_err}")


def fetch_fred_series(series_id: str, observation_start: str) -> Dict[str, Any]:
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": observation_start,
    }
    return get_json_with_retry(BASE_URL, params=params,
                               headers={"User-Agent": "databricks-lakehouse-demo/1.0"})


def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation",
                  "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])


def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")

# COMMAND ----------

# ---------------- Read watermarks (only used in incremental mode) ----------------
watermarks = {}
if MODE == "incremental":
    wm_df = spark.sql("""
        SELECT series_key, last_obs_date
        FROM canada_business.bronze.ingest_watermarks
        WHERE source = 'fred'
    """).toPandas()
    for _, row in wm_df.iterrows():
        if row["last_obs_date"] is not None:
            watermarks[row["series_key"]] = row["last_obs_date"]
    print(f"Loaded watermarks for {len(watermarks)} series")

# COMMAND ----------

# ---------------- Resolve paths ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)
data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

ts = run_ts_utc()
print(f"Run ts: {ts} | MODE: {MODE} | Series: {len(ALL_SERIES)}")

# COMMAND ----------

# ---------------- Ingest ----------------
manifest = {
    "run_ts": ts,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "bronze_base": bronze_base,
    "mode": MODE,
    "series": []
}

for series_id, desc in ALL_SERIES.items():

    # ---- Determine observation_start based on MODE ----
    if MODE == "backfill":
        obs_start = BACKFILL_START
        run_mode = "backfill"
    elif series_id in watermarks:
        obs_start = (watermarks[series_id] - dt.timedelta(days=OVERLAP_DAYS)).isoformat()
        run_mode = "incremental"
    else:
        # First incremental run for this series — backfill it
        obs_start = BACKFILL_START
        run_mode = "backfill"

    print(f"⏳ {series_id} [{run_mode}] from {obs_start}")

    payload = fetch_fred_series(series_id, observation_start=obs_start)
    obs_count = len(payload.get("observations", []))

    if obs_count == 0:
        print(f"  ⏭️ No observations, skipping")
        continue

    out_dir = join_uri(data_base, series_id)
    dbutils.fs.mkdirs(out_dir)

    # In backfill mode, delete old files first (clean slate)
    if MODE == "backfill":
        try:
            for f in dbutils.fs.ls(out_dir):
                if f.name.endswith(".json"):
                    dbutils.fs.rm(f.path)
            print(f"  🗑️ Cleaned old files for {series_id}")
        except Exception:
            pass  # directory might be empty

    out_file = join_uri(out_dir, f"{series_id}_observations_{ts}.json")
    dbutils.fs.put(out_file, json.dumps(payload, indent=2), overwrite=False)
    print(f"  ✅ {obs_count} obs -> {out_file}")

    # Find max observation date
    max_date = max(
        o["date"] for o in payload["observations"]
        if o.get("date") and o["date"] != "."
    )

    # Update watermark
    spark.sql(f"""
        MERGE INTO canada_business.bronze.ingest_watermarks AS t
        USING (SELECT
            'fred' AS source, '{series_id}' AS series_key,
            DATE('{max_date}') AS last_obs_date, '{ts}' AS last_run_ts,
            {obs_count} AS rows_written, current_timestamp() AS updated_at
        ) AS s
        ON t.source = s.source AND t.series_key = s.series_key
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    manifest["series"].append({
        "series_id": series_id, "description": desc,
        "mode": run_mode, "observation_start": obs_start,
        "rows": obs_count, "max_date": max_date, "raw_file": out_file
    })

manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

print(f"\n✅ FRED ingest complete ({MODE}): {len(manifest['series'])} series")
print("Manifest:", manifest_path)